In [ ]:
!unzip Question_Image_Retrieval.zip

In [3]:
%ls

Image/                        Question_Text_Image_with_Retrieval.json
Question_Image_Retrieval.zip  sample_data/


### import libraries

In [4]:
import os
from google import genai
from google.genai import types
from google.colab import userdata
import time
import json
import re

### get this token from google AI studio and set it in your environment: https://aistudio.google.com/

In [5]:
os.environ["GEMINI_API_KEY"] = userdata.get('Gemeni_API_KEY')

### utility function

In [6]:
def _infer_mime_type_from_path(path: str) -> str:
    """
    Helper to guess mime_type from file extension.
    Extend this if needed (png, jpg, jpeg, gif, etc).
    """
    lower = path.lower()
    if lower.endswith(".jpg") or lower.endswith(".jpeg"):
        return "image/jpeg"
    elif lower.endswith(".png"):
        return "image/png"
    elif lower.endswith(".gif"):
        return "image/gif"
    else:
        # default fallback
        return "application/octet-stream"

In [7]:
def extract_answer_choice(text):
    """Extract answer choice (1-4) from model output."""
    if text is None:
        return None
    match = re.search(r'[1-4]', text)
    return int(match.group()) if match else None

### generative model function

In [8]:
SYSTEM_PROMPT_IRANIAN_FIGURES = """
شما یک مدل هوش مصنوعی هستید که تخصص‌تان در پاسخ‌دهی درباره مشاهیر ایرانی است.

اگر در زمینه اطلاعات مرتبط برای پاسخ به سوال وجود دارد از آن استفاده کن و اگر مرتبط نیست یا وجود ندارد از دانش عمومی خودت استفاده کن.

سوال 4 گزینه است لطفا فقط یکی از گزینه ها را انتخاب کن و فقط در یک کلمه یک عدد از 1 تا 4 انتخاب کن.
"""

def call_generative_model(
    model_name: str,
    input_text: str | None,
    input_image_path: str | None,
    retrieved_texts: list[str] | None = None,
    retrieved_images_paths: list[str] | None = None,
    temperature: float = 0.25,
    top_p: float = 0.5,
    max_output_tokens: int = 256,
    thinking_budget: int = 0
) -> str:
    """
    Calls a Gemini generative model with optional text & image context,
    incorporating retrieved texts/images, and optional main image + text.

    Either or both of input_text / input_image_path can be None,
    similarly retrieved_texts or retrieved_images_paths can be None.

    Args:
        model_name: name of the Gemini model, e.g. "gemini-2.5-flash"
        input_text: main user prompt / query (may be None if prompt only image)
        input_image_path: path to image file for main prompt (may be None)
        retrieved_texts: list of text strings retrieved by retriever (or None)
        retrieved_images_paths: list of image file paths retrieved as context (or None)
        temperature, top_p, max_output_tokens, thinking_budget: config params

    Returns:
        Generated text from the model.

    Raises:
        ValueError if both input_text and input_image_path are None (no prompt).
    """

    # Basic checks
    if input_text is None and input_image_path is None:
        raise ValueError("At least one of input_text or input_image_path must be provided for prompt.")

    client = genai.Client()

    contents: list[types.Part | str] = []

    # 1. Retrieved text context
    if retrieved_texts:
        # Format nicely, maybe separate them
        # text_context = "Context from retrieved texts:\n" + "\n---\n".join(retrieved_texts)
        text_context = "زمینه مرتبط از متون بازیابی‌شده:\n" + "\n---\n".join(retrieved_texts)
        contents.append(text_context)

    # 2. Retrieved images as context
    if retrieved_images_paths:
        for img_path in retrieved_images_paths:
            with open(img_path, "rb") as f:
                img_bytes = f.read()
            part = types.Part.from_bytes(
                data=img_bytes,
                mime_type=_infer_mime_type_from_path(img_path)
            )
            contents.append(part)

    # 3. Main input image
    if input_image_path:
        with open(input_image_path, "rb") as f:
            main_img_bytes = f.read()
        main_img_part = types.Part.from_bytes(
            data=main_img_bytes,
            mime_type=_infer_mime_type_from_path(input_image_path)
        )
        contents.append(main_img_part)

    # 4. Main input text
    if input_text:
        contents.append(input_text)

    # 5. Generation config
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_output_tokens,
        thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget),
        system_instruction=SYSTEM_PROMPT_IRANIAN_FIGURES
    )

    # 6. Call Gemini
    response = client.models.generate_content(
        model=model_name,
        contents=contents,
        config=config
    )

    time.sleep(8)

    return response.text


## evaluate multimodal without retrivial

In [9]:
with open("/content/Question_Text_Image_with_Retrieval.json", "r") as f:
    multi_modal_questions = json.load(f)

In [13]:
correct_questions = []
wrong_questions = []
total = 0
correct = 0

for multi_modal_question in multi_modal_questions:
    question = multi_modal_question["question"]
    answer_idx = multi_modal_question["answer"]
    image_path = multi_modal_question["image_path"]
    image_path = "/content" + image_path
    question_text = ""
    for q in question:
        question_text += q + "\n"

    ans = call_generative_model(model_name="gemini-2.5-flash",
                            input_text=question_text,
                            input_image_path=image_path
                            )

    ans_int = extract_answer_choice(ans)

    if ans_int == answer_idx:
        correct += 1
        correct_questions.append(multi_modal_question)
    else:
        wrong_questions.append(multi_modal_question)

    total += 1

    print(f"index: {total}, total: {70}, acc: {correct / total}")

accuracy = correct / total
print(accuracy)

index: 1, total: 70, acc: 0.0
index: 2, total: 70, acc: 0.5
index: 3, total: 70, acc: 0.3333333333333333
index: 4, total: 70, acc: 0.25
index: 5, total: 70, acc: 0.4
index: 6, total: 70, acc: 0.5
index: 7, total: 70, acc: 0.5714285714285714
index: 8, total: 70, acc: 0.625
index: 9, total: 70, acc: 0.6666666666666666
index: 10, total: 70, acc: 0.7
index: 11, total: 70, acc: 0.6363636363636364
index: 12, total: 70, acc: 0.6666666666666666
index: 13, total: 70, acc: 0.6153846153846154
index: 14, total: 70, acc: 0.6428571428571429
index: 15, total: 70, acc: 0.6
index: 16, total: 70, acc: 0.5625
index: 17, total: 70, acc: 0.5294117647058824
index: 18, total: 70, acc: 0.5555555555555556
index: 19, total: 70, acc: 0.5789473684210527
index: 20, total: 70, acc: 0.6
index: 21, total: 70, acc: 0.6190476190476191
index: 22, total: 70, acc: 0.6363636363636364
index: 23, total: 70, acc: 0.6521739130434783
index: 24, total: 70, acc: 0.625
index: 25, total: 70, acc: 0.64
index: 26, total: 70, acc: 0.6

### save results

In [14]:
with open("correct_questions_without_RAG.json", "w", encoding="utf-8") as f:
    json.dump(correct_questions, f, ensure_ascii=False, indent=4)

In [15]:
with open("wrong_questions_without_RAG.json", "w", encoding="utf-8") as f:
    json.dump(wrong_questions, f, ensure_ascii=False, indent=4)